In [ ]:
"""
WC-VC（Weighted Cascade Viral Centrality）計算スクリプト
=========================================================
network.txt の元ノードID（9090999...等）を 0-based index に変換し、
WC モデルで P 行列を構築して v = (I-P)^{-1} · 1 を解く。

使い方:
    python wc_vc.py network.txt

network.txt の形式:
    <from_id> <to_id>   （1行1エッジ、空白区切り）
    例: 9090999 3847291
"""

import sys
import time
import numpy as np
import networkx as nx
from scipy.sparse import lil_matrix, csr_matrix, eye
from scipy.sparse.linalg import spsolve, eigs


# ─────────────────────────────────────────────
# 対象 IP（内部 index）
# ─────────────────────────────────────────────
IP_INDICES = [38598, 28025, 9199, 27638, 12605,
              69023, 79973, 68985, 51867, 33649]


def load_graph_and_mapping(txt_path: str):
    """
    network.txt を読み込み、元IDを 0-based index に変換した
    有向グラフと対応表を返す。

    graph_lib の変換順序を再現するために「登場順（insertion order）」
    でインデックスを付与する。もし graph_lib が昇順ソートなら
    sort_ids=True に変更すること。

    Returns
    -------
    G       : nx.DiGraph  （ノード番号 = 0-based index）
    orig_ids: list         （index → 元ID の逆引きリスト）
    id_map  : dict         （元ID → index の正引き辞書）
    """
    print(f"[1] Reading {txt_path} ...")
    t0 = time.time()

    raw_edges = []
    seen = {}          # 元ID → index （登場順）

    def get_idx(orig_id):
        if orig_id not in seen:
            seen[orig_id] = len(seen)
        return seen[orig_id]

    with open(txt_path) as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            parts = line.split()
            u_orig, v_orig = int(parts[0]), int(parts[1])
            u_idx = get_idx(u_orig)
            v_idx = get_idx(v_orig)
            raw_edges.append((v_idx, u_idx))

    # 逆引きリスト（index → 元ID）
    orig_ids = [None] * len(seen)
    for orig_id, idx in seen.items():
        orig_ids[idx] = orig_id

    G = nx.DiGraph()
    G.add_nodes_from(range(len(seen)))
    G.add_edges_from(raw_edges)

    print(f"    ノード数: {G.number_of_nodes():,}")
    print(f"    エッジ数: {G.number_of_edges():,}")
    print(f"    所要時間: {time.time()-t0:.1f}s")
    return G, orig_ids, seen


def show_ip_mapping(ip_indices, orig_ids):
    """IP の内部 index と元ID の対応を表示"""
    print("\n[2] IP の index ↔ 元ID 対応")
    print(f"    {'内部 index':>12}  {'元ID':>15}")
    print("    " + "-" * 30)
    for idx in ip_indices:
        if idx < len(orig_ids):
            print(f"    {idx:>12}  {orig_ids[idx]:>15}")
        else:
            print(f"    {idx:>12}  {'範囲外':>15}")


def build_wc_matrix(G: nx.DiGraph) -> csr_matrix:
    """
    WC モデルの疎行列 P を構築する。

    WC の定義:  p_{u→v} = 1 / d_in(v)
    行列表現:   P[v, u] = 1 / d_in(v)  （v が被影響ノード、u が影響元）

    つまり P の列和 = 1（各入次数で正規化済み）。
    """
    print("\n[3] WC 行列を構築中 ...")
    t0 = time.time()
    n = G.number_of_nodes()

    # lil_matrix は行方向の書き込みが速い
    P = lil_matrix((n, n), dtype=np.float64)

    for v in G.nodes():
        d_in = G.in_degree(v)
        if d_in == 0:
            continue
        weight = 0.5 / d_in
        for u in G.predecessors(v):
            P[v, u] = weight   # P[被影響, 影響元]

    P_csr = P.tocsr()
    print(f"    非ゼロ要素数: {P_csr.nnz:,}")
    print(f"    所要時間: {time.time()-t0:.1f}s")
    return P_csr


def check_convergence(P_csr: csr_matrix) -> float:
    """
    最大固有値 |λ|_max を計算し収束条件を確認。
    (I-P)^{-1} の収束には |λ|_max < 1 が必要。
    """
    print("\n[4] 収束条件の確認（最大固有値）...")
    try:
        vals, _ = eigs(P_csr.astype(complex), k=1, which='LM', maxiter=5000)
        lam_max = float(np.max(np.abs(vals)))
        status = "✓ 収束（VC 計算可能）" if lam_max < 1.0 else "✗ 発散（要注意）"
        print(f"    |λ|_max = {lam_max:.6f}  {status}")
        return lam_max
    except Exception as e:
        print(f"    固有値計算エラー（スキップ）: {e}")
        return -1.0


def compute_vc_for_ips(P_csr: csr_matrix, ip_indices: list) -> dict:
    """
    指定した IP ノードについて VC を計算する。

    v = (I - P)^{-1} · 1  を解くのではなく、
    各 IP i について e_i^T (I - P)^{-T} · 1 = VC(i) を求める。

    = (I - P^T) x = e_i  を解いて x の和を取る方法が効率的。

    または列 i だけ必要なら:
        (I - P) v = e_i  →  v = (I-P)^{-1} e_i
        VC(i) = sum(v) - 1   ← i 自身を除く

    ここでは後者の方法を使う（IP は10個のみ）。
    """
    print(f"\n[5] VC を計算中（{len(ip_indices)} 個の IP）...")
    n = P_csr.shape[0]
    I_minus_P = eye(n, format='csr') - P_csr
    results = {}

    for ip in ip_indices:
        if ip >= n:
            print(f"    IP {ip}: 範囲外（スキップ）")
            continue
        t0 = time.time()

        # e_ip: ip 番目だけ 1 の単位ベクトル
        e_ip = np.zeros(n)
        e_ip[ip] = 1.0

        # (I - P) · v = e_ip  を解く
        # v[j] = "ip から j への到達に関わる期待値の寄与"
        v = spsolve(I_minus_P, e_ip)

        # VC = v の全和 - 1（ip 自身を除く）
        vc = float(v.sum()) - 1.0
        elapsed = time.time() - t0
        results[ip] = vc
        print(f"    IP {ip:>6}: VC = {vc:>12.2f}  ({elapsed:.2f}s)")

    return results


def compare_with_xlsx(vc_results: dict, orig_ids: list):
    """
    WC-VC と xlsx の実測値（全体像）を並べて表示する。
    xlsx の値は直接入力（実際の分析では CSV 読み込みに変更可）。
    """
    # xlsx から取得した全体 msg 外部行動者数（実測 VC_X）
    xlsx_vc_x = {
        38598: 4948.19, 28025: 1470.74, 9199:  100.00,
        27638:   67.98, 12605:   11.52, 69023:  47.24,
        79973:   11.12, 68985:    4.28, 51867:   7.67,
        33649:    0.34,
    }

    print("\n[6] WC-VC vs 実測 VC_X（外部行動者数）比較")
    print(f"    {'IP':>8}  {'元ID':>15}  {'WC-VC':>12}  {'実測VC_X':>12}  {'比率':>8}")
    print("    " + "-" * 65)

    for ip in sorted(vc_results.keys(), key=lambda x: -vc_results[x]):
        orig = orig_ids[ip] if ip < len(orig_ids) else "?"
        wc   = vc_results[ip]
        real = xlsx_vc_x.get(ip, float('nan'))
        ratio = wc / real if real > 0 else float('nan')
        print(f"    {ip:>8}  {orig:>15}  {wc:>12.2f}  {real:>12.2f}  {ratio:>8.3f}")

    # Spearman 相関
    from scipy.stats import spearmanr
    ips_common = [ip for ip in vc_results if ip in xlsx_vc_x]
    if len(ips_common) >= 3:
        wc_vals   = [vc_results[ip] for ip in ips_common]
        real_vals = [xlsx_vc_x[ip]  for ip in ips_common]
        rho, pval = spearmanr(wc_vals, real_vals)
        print(f"\n    Spearman 相関係数 ρ = {rho:.4f}  (p = {pval:.4f})")
        if abs(rho) > 0.7:
            print("    → WC-VC は実測 VC_X の良い代理変数")
        elif abs(rho) > 0.4:
            print("    → 中程度の相関（他の指標との組み合わせを検討）")
        else:
            print("    → 相関が低い（動的要因が支配的）")


# ─────────────────────────────────────────────
# メイン
# ─────────────────────────────────────────────
def main():
    txt_path = "/Users/hongseokyeong/Desktop/diffusion_seokyeong/test/twitter_dataset.txt"
    G, orig_ids, id_map = load_graph_and_mapping(txt_path)
    show_ip_mapping(IP_INDICES, orig_ids)

    P_csr = build_wc_matrix(G)
    lam   = check_convergence(P_csr)

    if lam >= 1.0:
        print("\n  ⚠ |λ|_max >= 1 のため (I-P)^{-1} が収束しない可能性があります。")
        print("    p スケーリング（P = P * 0.5 など）を検討してください。")
        # オプション：p=0.5 をかけて収束させる
        # P_csr = P_csr * 0.5

    vc_results = compute_vc_for_ips(P_csr, IP_INDICES)
    compare_with_xlsx(vc_results, orig_ids)

    # 結果を CSV に保存
    import csv
    out_path = "wc_vc_results.csv"
    with open(out_path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["ip_index", "orig_id", "wc_vc"])
        for ip, vc in vc_results.items():
            orig = orig_ids[ip] if ip < len(orig_ids) else ""
            writer.writerow([ip, orig, round(vc, 4)])
    print(f"\n[7] 結果を {out_path} に保存しました。")


if __name__ == "__main__":
    main()

[1] Reading /Users/hongseokyeong/Desktop/diffusion_seokyeong/test/twitter_dataset.txt ...
    ノード数: 81,306
    エッジ数: 1,768,149
    所要時間: 2.7s

[2] IP の index ↔ 元ID 対応
        内部 index              元ID
    ------------------------------
           38598         17756596
           28025         29406557
            9199         14348594
           27638         71271236
           12605         18030840
           69023          6604072
           79973        177046670
           68985         44682842
           51867        178593058
           33649         60443269

[3] WC 行列を構築中 ...
    非ゼロ要素数: 1,768,149
    所要時間: 1.7s

[4] 収束条件の確認（最大固有値）...
    |λ|_max = 1.000000  ✗ 発散（要注意）

  ⚠ |λ|_max >= 1 のため (I-P)^{-1} が収束しない可能性があります。
    p スケーリング（P = P * 0.5 など）を検討してください。

[5] VC を計算中（10 個の IP）...


In [4]:
import time
import networkx as nx
# ===== 変換の整合性を検証 =====
IP_INDICES = [38598, 28025, 9199, 27638, 12605,
              69023, 79973, 68985, 51867, 33649]


def load_graph_and_mapping(txt_path: str):
    """
    network.txt を読み込み、元IDを 0-based index に変換した
    有向グラフと対応表を返す。

    graph_lib の変換順序を再現するために「登場順（insertion order）」
    でインデックスを付与する。もし graph_lib が昇順ソートなら
    sort_ids=True に変更すること。

    Returns
    -------
    G       : nx.DiGraph  （ノード番号 = 0-based index）
    orig_ids: list         （index → 元ID の逆引きリスト）
    id_map  : dict         （元ID → index の正引き辞書）
    """
    print(f"[1] Reading {txt_path} ...")
    t0 = time.time()

    raw_edges = []
    seen = {}          # 元ID → index （登場順）

    def get_idx(orig_id):
        if orig_id not in seen:
            seen[orig_id] = len(seen)
        return seen[orig_id]

    with open(txt_path) as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            parts = line.split()
            u_orig, v_orig = int(parts[0]), int(parts[1])
            u_idx = get_idx(u_orig)
            v_idx = get_idx(v_orig)
            raw_edges.append((u_idx, v_idx))

    # 逆引きリスト（index → 元ID）
    orig_ids = [None] * len(seen)
    for orig_id, idx in seen.items():
        orig_ids[idx] = orig_id

    G = nx.DiGraph()
    G.add_nodes_from(range(len(seen)))
    G.add_edges_from(raw_edges)

    print(f"    ノード数: {G.number_of_nodes():,}")
    print(f"    エッジ数: {G.number_of_edges():,}")
    print(f"    所要時間: {time.time()-t0:.1f}s")
    return G, orig_ids, seen

txt_path = "/Users/hongseokyeong/Desktop/diffusion_seokyeong/test/twitter_dataset.txt"
G, orig_ids, id_map = load_graph_and_mapping(txt_path)
# 1. txt ファイルの先頭10行を生で確認
print("=== txt 生データ（先頭10行）===")
with open(txt_path) as f:
    for i, line in enumerate(f):
        if i >= 10: break
        print(repr(line.strip()))

# 2. マッピングの先頭・末尾を確認
print(f"\n=== index → 元ID（先頭10件）===")
for i in range(min(10, len(orig_ids))):
    print(f"  index {i:>6} → 元ID {orig_ids[i]}")

print(f"\n=== index → 元ID（末尾5件）===")
for i in range(max(0, len(orig_ids)-5), len(orig_ids)):
    print(f"  index {i:>6} → 元ID {orig_ids[i]}")

# 3. IP の元ID を確認
print(f"\n=== IP の index ↔ 元ID ===")
for ip in IP_INDICES:
    orig = orig_ids[ip] if ip < len(orig_ids) else "範囲外"
    print(f"  index {ip:>6} → 元ID {orig}")

# 4. 総ノード数・エッジ数
print(f"\nノード総数: {len(orig_ids):,}")
print(f"エッジ総数: {G.number_of_edges():,}")

# 5. IP の次数確認（おかしい IP がないか）
print(f"\n=== IP の次数 ===")
print(f"  {'index':>8}  {'元ID':>15}  {'in-degree':>10}  {'out-degree':>11}")
for ip in IP_INDICES:
    if ip < len(orig_ids):
        orig = orig_ids[ip]
        d_in  = G.in_degree(ip)
        d_out = G.out_degree(ip)
        print(f"  {ip:>8}  {orig:>15}  {d_in:>10}  {d_out:>11}")

# 6. graph_lib が「登場順」か「昇順ソート」かを判定
# → orig_ids が昇順になっていれば graph_lib は昇順採番
is_sorted = all(orig_ids[i] <= orig_ids[i+1] for i in range(min(1000, len(orig_ids)-1)))
print(f"\n元ID の並び順が昇順か: {is_sorted}")
print("（True なら sort 採番、False なら登場順採番）")

[1] Reading /Users/hongseokyeong/Desktop/diffusion_seokyeong/test/twitter_dataset.txt ...
    ノード数: 81,306
    エッジ数: 1,768,149
    所要時間: 2.5s
=== txt 生データ（先頭10行）===
'214328887 34428380'
'17116707 28465635'
'380580781 18996905'
'221036078 153460275'
'107830991 17868918'
'151338729 222261763'
'19705747 34428380'
'222261763 88323281'
'19933035 149538028'
'158419434 17434613'

=== index → 元ID（先頭10件）===
  index      0 → 元ID 214328887
  index      1 → 元ID 34428380
  index      2 → 元ID 17116707
  index      3 → 元ID 28465635
  index      4 → 元ID 380580781
  index      5 → 元ID 18996905
  index      6 → 元ID 221036078
  index      7 → 元ID 153460275
  index      8 → 元ID 107830991
  index      9 → 元ID 17868918

=== index → 元ID（末尾5件）===
  index  81301 → 元ID 100857258
  index  81302 → 元ID 104883575
  index  81303 → 元ID 100946848
  index  81304 → 元ID 99943653
  index  81305 → 元ID 104885715

=== IP の index ↔ 元ID ===
  index  38598 → 元ID 17756596
  index  28025 → 元ID 29406557
  index   9199 → 元ID 143485